In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.sparse as sp
import zarr

# Input folders for both Xenium runs
BASE_5K = Path("C:/Users/ntpar/Downloads/Xenium_Prime_Human_Ovary_Cancer_FF_xe_outs")
BASE_V1 = Path("C:/Users/ntpar/Downloads/Xenium_V1_Human_Ovary_Cancer_FF_xe_outs")

c:\Users\ntpar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ntpar\AppData\Local\Programs\Python\Python313\Lib\site-packages\spatialdata\_core\query\relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
c:\Users\ntpar\AppData\Local\Programs\Python\Python313\Lib\site-packages\spatialdata\_core\query\relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_exclusive_join_spatialelement_table)
c:\Users\ntpar\AppData\Local\Programs\Python\Pytho

## Creating the H5 file

In [ ]:
def pick_root(run_dir: str | Path) -> Path:
    p = Path(run_dir)
    if (p / "experiment.xenium").exists():
        return p
    if (p / "outs" / "experiment.xenium").exists():
        return p / "outs"
    return p

# 10x cell_id decoding: uint32 pair -> shifted hex string
_hex_to_shift = str.maketrans("0123456789abcdef", "abcdefghijklmnop")

def decode_cell_ids(cell_id_2col_uint32: np.ndarray) -> np.ndarray:
    arr = np.asarray(cell_id_2col_uint32)
    prefix = arr[:, 0].astype(np.uint32)
    suffix = arr[:, 1].astype(np.uint32)

    out = np.empty(prefix.shape[0], dtype=object)
    for i, (pre, suf) in enumerate(zip(prefix, suffix)):
        hx = f"{int(pre):08x}"
        out[i] = f"{hx.translate(_hex_to_shift)}-{int(suf)}"
    return out.astype(str)

def normalize_feature_types(feature_types: np.ndarray) -> np.ndarray:
    normalized = []
    for feature_type in np.asarray(feature_types, dtype=str):
        value = feature_type.strip().lower()
        if value == "gene":
            normalized.append("Gene Expression")
        elif value in {"antibody", "antibody_capture"}:
            normalized.append("Antibody Capture")
        elif value in {"crispr", "crispr_guide_capture"}:
            normalized.append("CRISPR Guide Capture")
        elif value in {"multiplexing_capture", "hashtag"}:
            normalized.append("Multiplexing Capture")
        else:
            normalized.append(feature_type)
    return np.asarray(normalized, dtype=str)

def write_10x_h5_v3(
    h5_path: Path,
    X_csc: sp.csc_matrix,
    barcodes: np.ndarray,
    feature_ids: np.ndarray,
    feature_names: np.ndarray,
    feature_types: np.ndarray,
 ) -> None:
    with h5py.File(h5_path.as_posix(), "w") as handle:
        matrix = handle.create_group("matrix")
        matrix.create_dataset("barcodes", data=np.asarray(barcodes, dtype="S"))
        matrix.create_dataset("data", data=X_csc.data.astype(np.int32))
        matrix.create_dataset("indices", data=X_csc.indices.astype(np.int32))
        matrix.create_dataset("indptr", data=X_csc.indptr.astype(np.int64))
        matrix.create_dataset("shape", data=np.array(X_csc.shape, dtype=np.int64))

        features = matrix.create_group("features")
        features.create_dataset("id", data=np.asarray(feature_ids, dtype="S"))
        features.create_dataset("name", data=np.asarray(feature_names, dtype="S"))
        features.create_dataset("feature_type", data=np.asarray(feature_types, dtype="S"))
        features.create_dataset("genome", data=np.asarray([""] * len(feature_names), dtype="S"))

def make_cell_feature_matrix_h5_from_zip(run_dir: str | Path, force_rebuild: bool = False) -> Path:
    root = pick_root(run_dir)

    zarr_zip = root / "cell_feature_matrix.zarr.zip"
    if not zarr_zip.exists():
        matches = list(root.rglob("cell_feature_matrix.zarr.zip"))
        if not matches:
            raise FileNotFoundError(f"No cell_feature_matrix.zarr.zip found under {root}")
        zarr_zip = matches[0]

    h5_path = root / "cell_feature_matrix.h5"
    if force_rebuild and h5_path.exists():
        h5_path.unlink()

    if h5_path.exists() and h5_path.stat().st_size == 0:
        h5_path.unlink()

    if h5_path.exists() and h5_path.stat().st_size > 0:
        return h5_path

    store = zarr.storage.ZipStore(zarr_zip.as_posix(), mode="r")
    try:
        group = zarr.open_group(store=store, mode="r")
        cell_features = group["cell_features"]
        attrs = cell_features.attrs.asdict()

        required = ["feature_keys", "feature_ids", "feature_types"]
        missing = [key for key in required if key not in attrs]
        if missing:
            raise KeyError(f"Missing attributes in cell_features: {missing}")

        feature_names = np.array(attrs["feature_keys"], dtype=str)
        feature_ids = np.array(attrs["feature_ids"], dtype=str)
        feature_types_raw = np.array(attrs["feature_types"], dtype=str)
        feature_types = normalize_feature_types(feature_types_raw)

        data = np.array(cell_features["data"])
        indices = np.array(cell_features["indices"])
        indptr = np.array(cell_features["indptr"])

        n_features = len(indptr) - 1
        cell_id_2col = np.array(cell_features["cell_id"])
        n_cells = cell_id_2col.shape[0]

        if feature_names.size != n_features:
            raise ValueError(
                f"Feature mismatch: indptr implies {n_features} features, attrs has {feature_names.size}"
            )

        barcodes = decode_cell_ids(cell_id_2col)

        cells_parquet = root / "cells.parquet"
        if cells_parquet.exists():
            table_ids = pd.read_parquet(cells_parquet, columns=["cell_id"])["cell_id"].astype(str).values
            sample_n = min(5000, len(table_ids), len(barcodes))
            match_rate = np.mean(table_ids[:sample_n] == barcodes[:sample_n])
            print(f"cell_id check vs cells.parquet (first {sample_n}): {match_rate*100:.2f}% match")
            if match_rate < 0.99:
                print("Example table vs decoded:", list(zip(table_ids[:5], barcodes[:5])))

        print("Feature type counts (raw):")
        print(pd.Series(feature_types_raw).value_counts(dropna=False).head(10))
        print("Feature type counts (normalized):")
        print(pd.Series(feature_types).value_counts(dropna=False).head(10))

        X_csr = sp.csr_matrix((data, indices, indptr), shape=(n_features, n_cells))
        X_csc = X_csr.tocsc()

        write_10x_h5_v3(h5_path, X_csc, barcodes, feature_ids, feature_names, feature_types)
        print("Wrote:", h5_path, "bytes:", h5_path.stat().st_size)
        return h5_path
    finally:
        store.close()

# Build (or rebuild) the H5 for this run
xenium_dir = BASE_V1
make_cell_feature_matrix_h5_from_zip(xenium_dir, force_rebuild=True)

cell_id check vs cells.parquet (first 5000): 100.00% match
Feature type counts before normalization:
gene                         477
negative_control_codeword     41
negative_control_probe        20
unassigned_codeword            3
aggregate_gene                 1
Name: count, dtype: int64
Feature type counts after normalization:
Gene Expression              477
negative_control_codeword     41
negative_control_probe        20
unassigned_codeword            3
aggregate_gene                 1
Name: count, dtype: int64
Wrote: C:\Users\ntpar\Downloads\Xenium_V1_Human_Ovary_Cancer_FF_xe_outs\cell_feature_matrix.h5 bytes: 191686286


WindowsPath('C:/Users/ntpar/Downloads/Xenium_V1_Human_Ovary_Cancer_FF_xe_outs/cell_feature_matrix.h5')

## Creating .parquet files

In [ ]:
def open_cells_zarr(root: Path):
    zarr_dir = root / "cells.zarr"
    zarr_zip = root / "cells.zarr.zip"
    if zarr_dir.exists():
        return zarr.open(zarr_dir.as_posix(), mode="r"), None
    if zarr_zip.exists():
        store = zarr.storage.ZipStore(zarr_zip.as_posix(), mode="r")
        return zarr.open_group(store=store, mode="r"), store
    raise FileNotFoundError(f"Missing {zarr_dir} and {zarr_zip}")

def make_cells_parquet(run_dir: str | Path, force_rebuild: bool = False) -> Path:
    root = pick_root(run_dir)
    out_path = root / "cells.parquet"

    if force_rebuild and out_path.exists():
        out_path.unlink()
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    group, store = open_cells_zarr(root)
    try:
        cell_id_str = decode_cell_ids(np.array(group["cell_id"]))
        cell_summary = np.array(group["cell_summary"])  # (N, 8)

        # Keep the fields commonly expected by xenium loaders
        cells_df = pd.DataFrame({
            "cell_id": cell_id_str,
            "x_centroid": cell_summary[:, 0],
            "y_centroid": cell_summary[:, 1],
            "cell_area": cell_summary[:, 2],
            "nucleus_centroid_x": cell_summary[:, 3],
            "nucleus_centroid_y": cell_summary[:, 4],
            "nucleus_area": cell_summary[:, 5],
            "z_level": cell_summary[:, 6],
            "nucleus_count": cell_summary[:, 7],
        })

        cells_df.to_parquet(out_path, index=False)
        return out_path
    finally:
        if store is not None:
            store.close()

# Create cells.parquet for the selected run
xenium_path = BASE_5K
print("Wrote:", make_cells_parquet(xenium_path, force_rebuild=False))

Wrote: C:\Users\ntpar\Downloads\Xenium_Prime_Human_Ovary_Cancer_FF_xe_outs\cells.parquet


In [ ]:
def make_boundaries_parquet(
    xenium_dir: str | Path,
    polygon_set_index: int,
    out_name: str,
    chunk_cells: int = 20_000,
    force_rebuild: bool = False,
 ) -> Path:
    root = pick_root(xenium_dir)
    out_path = root / out_name

    if force_rebuild and out_path.exists():
        out_path.unlink()
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    group, store = open_cells_zarr(root)
    try:
        all_cell_ids = decode_cell_ids(np.array(group["cell_id"]))
        polygon_set = group["polygon_sets"][str(polygon_set_index)]

        cell_index = np.array(polygon_set["cell_index"]).astype(np.int64)
        num_vertices = np.array(polygon_set["num_vertices"]).astype(np.int64)
        vertices = np.array(polygon_set["vertices"])

        n_rows = cell_index.shape[0]
        identity_mapping = np.all(cell_index[: min(1000, n_rows)] == np.arange(min(1000, n_rows)))

        def label_ids_for_rows(start: int, end: int, idx_chunk: np.ndarray) -> np.ndarray:
            if identity_mapping:
                return (idx_chunk + 1).astype(np.int32)
            return (np.arange(start, end) + 1).astype(np.int32)

        writer = None
        for start in range(0, n_rows, chunk_cells):
            end = min(start + chunk_cells, n_rows)
            idx_chunk = cell_index[start:end]
            ids_chunk = all_cell_ids[idx_chunk]
            nv_chunk = num_vertices[start:end]
            vertices_chunk = vertices[start:end]
            labels_chunk = label_ids_for_rows(start, end, idx_chunk)

            xs_list = []
            ys_list = []
            cell_ids_list = []
            label_ids_list = []

            for i in range(end - start):
                n_vertices = int(nv_chunk[i])
                if n_vertices <= 0:
                    continue
                xy = vertices_chunk[i, : 2 * n_vertices].reshape(n_vertices, 2)
                xs_list.append(xy[:, 0])
                ys_list.append(xy[:, 1])
                cell_ids_list.append(np.repeat(ids_chunk[i], n_vertices))
                label_ids_list.append(np.repeat(labels_chunk[i], n_vertices))

            if not xs_list:
                continue

            table = pa.Table.from_pydict({
                "cell_id": np.concatenate(cell_ids_list).astype(str),
                "vertex_x": np.concatenate(xs_list).astype(np.float32),
                "vertex_y": np.concatenate(ys_list).astype(np.float32),
                "label_id": np.concatenate(label_ids_list).astype(np.int32),
            })

            if writer is None:
                writer = pq.ParquetWriter(out_path.as_posix(), table.schema)
            writer.write_table(table)

        if writer is not None:
            writer.close()

        return out_path
    finally:
        if store is not None:
            store.close()

# polygon_sets/0 = nucleus and polygon_sets/1 = cell
xenium_path = BASE_5K
print("Wrote:", make_boundaries_parquet(xenium_path, 0, "nucleus_boundaries.parquet"))
print("Wrote:", make_boundaries_parquet(xenium_path, 1, "cell_boundaries.parquet"))

Wrote: C:\Users\ntpar\Downloads\Xenium_Prime_Human_Ovary_Cancer_FF_xe_outs\nucleus_boundaries.parquet
Wrote: C:\Users\ntpar\Downloads\Xenium_Prime_Human_Ovary_Cancer_FF_xe_outs\cell_boundaries.parquet


In [ ]:
def build_transcripts_parquet_from_zarr(
    xenium_dir: str | Path,
    batch_rows: int = 1_000_000,
    force_rebuild: bool = False,
 ) -> Path:
    root = pick_root(xenium_dir)
    zarr_zip = root / "transcripts.zarr.zip"
    if not zarr_zip.exists():
        raise FileNotFoundError(f"Missing {zarr_zip}")

    out_dir = root / "transcripts.parquet"
    out_dir.mkdir(exist_ok=True)

    if force_rebuild:
        for old_part in out_dir.glob("part-*.parquet"):
            old_part.unlink()

    store = zarr.storage.ZipStore(zarr_zip.as_posix(), mode="r")
    try:
        group = zarr.open_group(store=store, mode="r")
        attrs = group.attrs.asdict()
        gene_names = np.array(attrs["gene_names"], dtype=object)
        fov_names = np.array(attrs["fov_names"], dtype=object)

        # grid level 0 contains all transcript points
        level = group["grids"]["0"]
        tile_keys = list(level.group_keys())

        schema = pa.schema([
            ("transcript_id", pa.int64()),
            ("cell_id", pa.string()),
            ("overlaps_nucleus", pa.int8()),
            ("feature_name", pa.string()),
            ("x_location", pa.float32()),
            ("y_location", pa.float32()),
            ("z_location", pa.float32()),
            ("qv", pa.float32()),
            ("fov_name", pa.string()),
            ("codeword_index", pa.int32()),
        ])

        part_idx = 0
        buffer = {name: [] for name in schema.names}
        buffered_rows = 0

        def flush() -> None:
            nonlocal part_idx, buffered_rows, buffer
            if buffered_rows == 0:
                return

            table = pa.Table.from_pydict(
                {
                    key: np.concatenate(values) if isinstance(values[0], np.ndarray) else np.array(values, dtype=object)
                    for key, values in buffer.items()
                },
                schema=schema,
            )
            pq.write_table(table, (out_dir / f"part-{part_idx:05d}.parquet").as_posix())
            part_idx += 1
            buffer = {name: [] for name in schema.names}
            buffered_rows = 0

        for tile_key in tile_keys:
            tile = level[tile_key]
            loc = np.array(tile["location"])
            qv = np.array(tile["quality_score"]).reshape(-1)
            gene_id = np.array(tile["gene_identity"]).reshape(-1).astype(np.int32)
            codeword_index = np.array(tile["codeword_identity"])[:, 0].astype(np.int32)

            raw_id = np.array(tile["id"])
            transcript_local_id = raw_id[:, 0].astype(np.int64)
            fov_idx = raw_id[:, 1].astype(np.int32)

            n = loc.shape[0]
            feature_name = np.empty(n, dtype=object)
            detected = gene_id != 65535
            feature_name[detected] = gene_names[gene_id[detected]]
            feature_name[~detected] = "UNASSIGNED"

            transcript_id = (fov_idx.astype(np.int64) << 32) + transcript_local_id
            fov_name = fov_names[fov_idx].astype(object)

            buffer["transcript_id"].append(transcript_id)
            buffer["cell_id"].append(np.array([""] * n, dtype=object))
            buffer["overlaps_nucleus"].append(np.zeros(n, dtype=np.int8))
            buffer["feature_name"].append(feature_name)
            buffer["x_location"].append(loc[:, 0].astype(np.float32))
            buffer["y_location"].append(loc[:, 1].astype(np.float32))
            buffer["z_location"].append(loc[:, 2].astype(np.float32))
            buffer["qv"].append(qv.astype(np.float32))
            buffer["fov_name"].append(fov_name)
            buffer["codeword_index"].append(codeword_index)

            buffered_rows += n
            if buffered_rows >= batch_rows:
                flush()

        flush()
        return out_dir
    finally:
        store.close()

# Build transcript parquet dataset
xenium_path = BASE_5K
out = build_transcripts_parquet_from_zarr(xenium_path)
print("Wrote parquet dataset at:", out)

Wrote parquet dataset at: C:\Users\ntpar\Downloads\Xenium_Prime_Human_Ovary_Cancer_FF_xe_outs\transcripts.parquet
